# Knockoffs

Knockoffs create synthetic "negative control" variables that mimic the correlation structure of the original features but are independent of the outcome. Comparing each feature to its knockoff enables FDR-controlled variable selection without p-values. 

See [Barber & Candès (2015)](https://projecteuclid.org/journals/annals-of-statistics/volume-43/issue-5/Controlling-the-false-discovery-rate-via-knockoffs/10.1214/15-AOS1337.full), [Candès et al. (2018)](https://academic.oup.com/jrsssb/article/80/3/551/7048447), which establish some of the core ideas and methods.

The knockoff framework offers a large number of approaches to generate knock-offs, which are often descibed in statistical terms about the assumptions they make about the feature or target distribution.

The two main families are:

 * Fixed-X : no assumption of feature distribution, but only for linear model with homoskadastic error (ie, classical OLS approach)
 * model-X : no restriction on the type of model, but you must know or estimate the distribution of features
 
Of these two, **model-X** is the most appropriate in most case, but it can be difficult to estimate X-distribution and to choose among the many option how we will generate the knockoffs.


[Cartier et al. (2026, *Briefings in Bioinformatics*)](https://academic.oup.com/bib/article/27/3/bbag148/8687371) investigates different approaches in a transcriptomics context and demonstrated that in the presence of a **linearly separable problem the method used to generate the knockoffs did not affect the results too much** and that in the presence of a more difficult problem no method gave good results anyway.



**Important note:** 

 * for some tasks, most famously GWAS, there exists dedicated knockoff generators: [knockoffzoom](https://msesia.github.io/knockoffzoom/tutorial.html)
 * if yu have plenty of data and computational resources, you can also take a look at [deepknockoffs](https://web.stanford.edu/group/candes/deep-knockoffs/tutorial-1.html)



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import SelectPercentile
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import AgglomerativeClustering

np.random.seed(123)


def drop_correlated_features( X , threshold = 0.9 ):
    """
    Args:
        - X (pd.DataFrame) : n,p feature matrix
        - threshold (float) : absolute correlation threshold group variables
    
    Returns:
        - pd.DataFrame : X with only the selected variables
        - dict : keys are the selected features , values are the list of features in the corresponding feature cluster 
    """
    
    corr_threshold = 0.9

    metric = 1 - X.corr().abs()

    HC = AgglomerativeClustering( n_clusters=None , metric='precomputed', linkage = 'single' , distance_threshold = (1-corr_threshold) )
    HC.fit(metric)

    variable_clusters = pd.Series( HC.labels_  , index = X.columns)

    cluster_to_features = variable_clusters.index.groupby(variable_clusters)

    ## keys are the selected feature in the cluster, values are the list of features in the cluster
    selected_features_to_features = { v[0]:list(v) for v in cluster_to_features.values() }

    return X.loc[:,selected_features_to_features.keys()] ,  selected_features_to_features 

## Data loading

In [ ]:
df_xpr = pd.read_csv("../data/TGCA_BRCA_expression_matrix.TPM.csv.gz", index_col=0)
df_clinical = pd.read_csv("../data/TGCA_BRCA_clinical_filtered.small.csv", index_col=0)
df_clinical = pd.get_dummies(df_clinical, drop_first=True)

y = df_clinical.poor_prognosis
X_xpr = df_xpr.loc[:, df_clinical.index].transpose()


## select the top1% variant genes
VT = SelectPercentile(score_func=lambda x, _: np.var(x, axis=0), percentile=1)
X = pd.DataFrame(VT.fit_transform(X_xpr), columns=VT.get_feature_names_out(), index=X_xpr.index)
X , features_to_features_cluster = drop_correlated_features( X , threshold = 0.9 )

## add other variables
X = pd.concat([df_clinical[["demographic.days_to_birth", "demographic.sex_at_birth_male"]], X], axis=1)
X.shape

## Demo: Basic knockoff pipeline

The main class of `knockpy` is `KnockoffFilter`, which wraps the whole knockoff generation and selection procedure.


Its main arguments are:
 * ksampler : The knockoff sampler to use in the knockoff filter.
    - 'gaussian': Gaussian Model-X knockoffs, the most common approach
    - 'fx': Fixed-X knockoffs.
    - 'artk': t-tailed Markov chain. Metropolis with a t-tailed distribution
    - 'metro': Generic metropolized knockoff sampler. Generic, but you have to provide your likelihood.
    - ...

 * `knockoff_kwargs={"method": "..."}`: sets up the methods used to generate the knockoff
    - 'mvr' (default): minimizes the varianced-based reconstructability (MVR) between the features and their knockoffs, preventing a feature statistic like a lasso or a randomforest from using the other features and knockoffs to reconstruct non-null features.
    - 'sdp' : minimize the mean absolute covariance (MAC) between features and their knockoffs. The "semidefinite program (SDP) is expected to be the most powerful as $p$ becomes large. The MAC approaches suffer from a tendency to identify some KO of non-null original features as important features in the model instead of the non-null features themselves."
    - 'ci' : Conditional Independence (CI) knockoffs 
    - ...


 * fstat : The feature statistic to use in the knockoff filter.
    - 'lasso' or 'lcd' (default): cross-validated lasso coefficients differences
    - 'mlr': the masked likelihood ratio (MLR) statistic
    - 'lsm': signed maximum of the lasso path statistic as in Barber and Candes 2015
            = largest lambda for which the feature has a non-null coefficient
    - 'dlasso': Cross-validated debiased lasso coefficients
    - 'ridge': Cross validated ridge coefficients
    - 'ols': Ordinary least squares coefficients
    - 'randomforest': A random forest with swap importances
    - ...


**IMPORTANT**: the pre-implemented `fstat` check if the target is binary to use a classifier or a regressor object. 

Both `ksampler` and `fstat` also accept custom objects which will be used in place of the pre-implemented options (e.g., giving your own model giving an importance ranking to `fstat`)


In [ ]:
from knockpy import KnockoffFilter

## scaling our features first
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

## run with default parameter: 
##  * gaussian knockoff generated with an MVR objective
##  * feature importance evaluated using cross-validated lasso coefficients differences
kf = KnockoffFilter(ksampler="gaussian", fstat="lasso")

Next we actually setup the knockoff selection with `kf.forward()`.

Main arguments:
 * X and y
 * fdr : the desired level of False Discovery Rate. 
 
 
 **NB:** [Cartier et al. (2026)](https://academic.oup.com/bib/article/27/3/bbag148/8687371) have shown that in a transcriptomics context KOs were conservative so we will pick a *high* value

In [ ]:
rejections = kf.forward(X=Xs, y=y.values, fdr=0.5)
selected = X.columns[rejections.astype(bool)]
print(f"Features selected by knockoffs at FDR=0.5: {list(selected)}")

In [ ]:
## kf.W contains the feature statistics (LCD in our case)
W_series = pd.Series(kf.W, index=X.columns)
print(f"Feature statistics W - {(W_series.abs() > 0).sum()}/{len(W_series)} non-zero")
print()
print("Top 10 features by W statistic:")
print(W_series.nlargest(10).to_string())

In [ ]:
## kf.Z : a ``2p``-dimensional array of feature and knockoff importances. The
##    first p coordinates correspond to features, the last p correspond
##    to knockoffs.
p = kf.X.shape[1]

# first p elements: original features
original_feature_importance = pd.Series( kf.Z[:p] , index = X.columns ).abs()

# remaining ones: knockoffs
knockoff_feature_importance = pd.Series( kf.Z[p:] , index = X.columns ).abs()

fig,ax = plt.subplots( figsize = (8,8) )

sns.scatterplot( x = original_feature_importance,
                 y = knockoff_feature_importance
               )

top10 = W_series.nlargest(10).index
for f in top10:
    ax.text( x = original_feature_importance[f],
              y = knockoff_feature_importance[f],
              s = f,
              rotation = 45, fontsize = 8
            )

ax.set_xlabel('original feature importance')
ax.set_ylabel('knockoff feature importance')
ax.axis('equal')
ax.axline( (0,0) , slope = 1 , color="black", linestyle = 'dashed', linewidth = 1)
ax.axhline( 0 , color="grey", linestyle = 'dotted' , linewidth = 1)
ax.axvline( 0 , color="grey", linestyle = 'dotted' , linewidth = 1)

In [ ]:
# kf.G contains correlation between all features (original and knockoffs)
# we can select some elements in it to look at different elements

fig,ax = plt.subplots( 2 , 2 , figsize = (12,12) )

sns.heatmap(kf.G[p:,p:], square=True, cmap="RdBu_r", vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False,
            ax = ax[0,0])
ax[0,0].set_title("original features correlation")

sns.heatmap(kf.G[:p,:p], square=True, cmap="RdBu_r", vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False,
            ax = ax[0,1])
ax[0,1].set_title("knockoff features correlation")


sns.heatmap(kf.G[p:,:p], square=True, cmap="RdBu_r", vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False,
            ax = ax[1,0])
ax[1,0].set_title("original and knockoff features correlation")


sns.histplot( np.apply_along_axis( lambda x : stats.pearsonr( x , y ).statistic , 
                                  axis = 0 , arr = kf.X ) , 
             ax = ax[1,1] , label = 'original' , binrange=[-1,1],binwidth=0.02 )
sns.histplot( np.apply_along_axis( lambda x : stats.pearsonr( x , y ).statistic , 
                                  axis = 0 , arr = kf.Xk ) , 
             ax = ax[1,1] , label = 'knockoff' , binrange=[-1,1],binwidth=0.02)
ax[1,1].set_xlabel("correlation with target")
ax[1,1].set_xlabel("correlation with target")


We can check specifically the correlation of features with their knock-offs:

In [ ]:
orig_vs_knock = pd.DataFrame({
    "corr_orig": np.diag( kf.G[p:,p:] ),
    "corr_knock": np.diag( kf.G[p:,:p] )
})
sns.histplot(data=orig_vs_knock.melt(var_name="type", value_name="correlation"),
             x="correlation", hue="type", bins=30, alpha=0.6)
plt.title("Self-correlation: original vs knockoff")

These are quite high-correlations with their knock-offs

---

We can look at a specific feature and its knock-off, for example the `"demographic.days_to_birth"`

In [ ]:

feature_index = list(X.columns).index('demographic.days_to_birth')

original_feature = kf.X[ : , feature_index ]
knockoff_feature = kf.Xk[ : , feature_index ]

sns.scatterplot(
    x = kf.G[ : , feature_index ],
    y = kf.G[ : , feature_index + p ]
)
plt.xlabel("correlation with original feature")
plt.ylabel("correlation with knockoff feature")

The two off-diagonal points correspond to the original feature and the knock-offs (correlated to 1.0 with iself, and <1.0 with the other one).

For the rest, we can see that the correlation of the features are the same to the original and the knockoff feature.

## knockoff - elements to consider

### element 1 : using a smaller feature set

Large feature set with heterogeneous distributions and correlation patterns can make good knock-off generation difficult.


In [ ]:
VT2 = SelectPercentile(score_func=lambda x, _: np.var(x, axis=0), percentile=0.2)

X_small = pd.DataFrame(VT2.fit_transform(X_xpr), columns=VT2.get_feature_names_out(), index=X_xpr.index)
X_small , features_to_features_cluster = drop_correlated_features( X_small , threshold = 0.9 )
X_small = pd.concat([df_clinical[["demographic.days_to_birth", "demographic.sex_at_birth_male"]], X_small], axis=1)
print(f"Reduced feature set: {X_small.shape[1]} features, {X_small.shape[0]} samples")

In [ ]:
%%time
X_s = StandardScaler().fit_transform(X_small)

kf_small = KnockoffFilter(ksampler="gaussian", fstat="lasso")
rej_small = kf_small.forward(X=X_s, y=y.values, fdr=0.5)

selected_small = X_small.columns[rej_small.astype(bool)]
print(f"Selected at FDR=0.5 (MLR): {len(selected_small)}")
print(list(selected_small))

In [ ]:
p = kf_small.X.shape[1]
orig_vs_knock = pd.DataFrame({
    "corr_orig": np.diag( kf_small.G[p:,p:] ),
    "corr_knock": np.diag( kf_small.G[p:,:p] )
})
sns.histplot(data=orig_vs_knock.melt(var_name="type", value_name="correlation"),
             x="correlation", hue="type", bins=30, alpha=0.6)
plt.title("Self-correlation: original vs knockoff")

### element 2 : using alternative models


If the method used to model the problem is inappropriate then the knock-offs are not going to help select interesting variables.

> NB: here that is likely not a problem because we get approximately the same performance from an logistic regression and a random forest

> For now, there is a small error in the code with `fstat='randomforest'` and binomial target ([issue](https://github.com/amspector100/knockpy/issues/11)), so we have to specify our model manually, but that also gives you an example on how this is done

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import knockpy

# Step 1: Initialize the RF object
RF = RandomForestClassifier(n_estimators=100)

# Step 2: Wrap it with a feature statistic
RF_fstat = knockpy.knockoff_stats.FeatureStatistic(model=RF)

# Step 3: Pass to a knockoff filter
kf_rf = KnockoffFilter(ksampler="gaussian", fstat=RF_fstat)


In [ ]:
rej_rf = kf_rf.forward(X=X_s, y=y, fdr=0.5)

selected_rf = X_small.columns[rej_rf.astype(bool)]
print(f"Selected at FDR=0.5 (MLR): {len(selected_rf)}")
print(list(selected_rf))

### element 3 : random instability

Knockoff generation is a random procedure, so running it twice may yield different results.



In [ ]:
%%time
X_s = StandardScaler().fit_transform(X_small)

kf_small = KnockoffFilter(ksampler="gaussian", fstat="lasso")

for i in range(3):
    rej_A = kf_small.forward(X=X_s, y=y.values, fdr=0.5)

    selected_A = X_small.columns[rej_A.astype(bool)]

    print(f'rep {i} : {list(selected_A)}')

Several KO aggregation methods exists, we will here follow the simple procedure from [Ren et al 2021](https://www.tandfonline.com/doi/full/10.1080/01621459.2021.1962720)

In [ ]:
%%time
X_s = StandardScaler().fit_transform(X_small)

kf_small = KnockoffFilter(ksampler="gaussian", fstat="lasso")

support = pd.Series(0, index=X_small.columns)

M = 20

for i in range(M):
    rej_A = kf_small.forward(X=X_s, y=y.values, fdr=0.5)

    support += rej_A.astype(int)/M


In [ ]:
support[support>0].sort_values(ascending=False)

So we see that demographic.days_to_birth beats its knockoff in most of the simulations, while only rarely beat their knockoff.


But this is already infringing upon ideas of the next chapter : stability selection

---
## Exercise: UCI Breast Cancer dataset

Apply knockoffs to the Breast Cancer Wisconsin dataset (diagnostic). Compare the results with the Boruta selection from Chapter 4.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X_bc, y_bc = data.data, data.target
X_bc.head()

**1.** Fit knockoffs with Gaussian MVR knockoffs and the lasso feature statistic (default parameters) at FDR=0.2. Which features are selected?

In [ ]:
# your code here

**2.** Try different FDR levels (0.05, 0.1, 0.2). How does the number of selected features change?

In [ ]:
# your code here

**3.** Print the overlap between the knockoff-selected and the features selected by Boruta from Chapter 4.

In [ ]:
# your comparison code here

**4. (Bonus)** Knockoff selection is stochastic — run the procedure 10 times and record which features are selected in each run. Which features are selected most consistently?

In [ ]:
# your code here

---
## Correction

In [ ]:
from knockpy import KnockoffFilter
from sklearn.preprocessing import StandardScaler

np.random.seed(456)
scaler_bc = StandardScaler()
X_bc_s = scaler_bc.fit_transform(X_bc.values)

kf_bc = KnockoffFilter(ksampler="gaussian", fstat="lasso")
rej_bc = kf_bc.forward(X=X_bc_s, y=y_bc.values, fdr=0.2)
selected_bc = X_bc.columns[rej_bc.astype(bool)]
print(f"Selected at FDR=0.2 ({len(selected_bc)} features):")
print(list(selected_bc))

In [ ]:
for fdr in [0.05, 0.1, 0.2]:
    np.random.seed(789)
    kf_tmp = KnockoffFilter(ksampler="gaussian", fstat="lasso")
    rej_tmp = kf_tmp.forward(X=X_bc_s, y=y_bc.values, fdr=fdr)
    print(f"FDR={fdr}: {rej_tmp.sum():.0f} features selected")

In [ ]:
# Features from chapter 4 (run it first to confirm)
# These are typical Boruta-confirmed features on this dataset:
boruta_typical = ["worst concave points", "worst perimeter", "worst area",
                  "mean concave points", "worst concavity", "worst radius"]

knockoff_set = set(selected_bc)
boruta_set = set(boruta_typical)
print(f"Knockoff selected: {sorted(knockoff_set)}")
print(f"Overlap with Boruta: {knockoff_set & boruta_set}")
print(f"In knockoff but not Boruta: {knockoff_set - boruta_set}")

In [ ]:
n_runs = 10
selection_counts = pd.Series(0, index=X_bc.columns)

for i in range(n_runs):
    np.random.seed(i * 10)
    kf_tmp = KnockoffFilter(ksampler="gaussian", fstat="lasso")
    rej_tmp = kf_tmp.forward(X=X_bc_s, y=y_bc.values, fdr=0.2)
    selection_counts += rej_tmp.astype(int)

print("Selection frequency across 10 runs:")
print(selection_counts[selection_counts > 0].sort_values(ascending=False).to_string())

## Annex:  KO e-value aggregation from https://academic.oup.com/jrsssb/article/86/1/122/7262479?login=false

In [ ]:
def get_Tm( W, alphakn , offset = 1):
    """ equation 15 from https://academic.oup.com/jrsssb/article/86/1/122/7262479?login=false """
    
    # equ 13
    W = kf_small.W
    ts = np.unique( np.sort( np.abs( W ) ) )

    for t in ts:

        # equ 13
        ratio = ( offset + ( W <= -t ).sum() ) / max(1,( W >= t ).sum())
        if ratio <= alphakn:
            return t
    
        # equ 15
        nb_selected = ( W >= t ).sum()
        if nb_selected < 1/alphakn:
            return t
    return np.inf
    
def get_evalues( W, Tm ):
    """ equation 14 from https://academic.oup.com/jrsssb/article/86/1/122/7262479?login=false """
    return W.shape[0] * (W>=Tm)/(1+(W<=-Tm).sum())

def evalues_benjamini_hochberg_threshold(E, alphaebh ):
    """ equation 9 from https://academic.oup.com/jrsssb/article/86/1/122/7262479?login=false """
    p = E.shape[0]

    E_sorted = np.sort(E)[::-1]
    comp = E_sorted >= p/( alphaebh * np.arange(1,p+1) )
    if comp.sum() == 0:
        return np.inf

    else:
        return E_sorted[comp][-1]


def KO_evalue_aggregation( X,y, fdr , M ):
    
    E = np.zeros( X.shape[1] )
    for i in range(M):

        rej_A = kf_small.forward(X=X, y=y.values, fdr=0.5)

        Tm = get_Tm( kf_small.W, alphakn=fdr/2 , offset = 0.1)
        E += get_evalues( kf_small.W , Tm ) / M

        
    threshold = evalues_benjamini_hochberg_threshold(E, alphaebh=fdr )
    
    E = pd.Series(E,index=X.columns)
    
    return E.index[E>=threshold] , E , threshold
    